### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="allstate_claims_severity",
    dataset_year="2016",
    domain_str="insurance",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/competitions/allstate-claims-severity",
    download_description="""
We use the train.csv from the Kaggle competition.

kaggle competitions download -c allstate-claims-severity -f train.csv && unzip train.csv.zip &&  rm train.csv.zip
mkdir -p local-data-warehouse/allstate_claims_severity && mv train.csv local-data-warehouse/allstate_claims_severity/
""",
    # References
    academic_reference_bibtex=r"""@misc{Ferguson2016AllstateClaimsSeverity,
  author = {Dana Ferguson and Meg Risdal and NoTrick and Sara R. Sillah and Tim Emmerling and Will Cukierski},
  title  = {Allstate Claims Severity},
  year   = {2016},
  howpublished = {\url{https://kaggle.com/competitions/allstate-claims-severity},
  note   = {Kaggle competition}
}
""",
    academic_reference_bibtex_key="Ferguson2016AllstateClaimsSeverity",
    license="Kaggle Competition Rules",
    data_tags=["IID", "Anonymized"],
    curation_comments="""
We start with the train.csv from Kaggle.

- The data has been anonymized.
- Top Kaggle solutions did not perform any relevant preprocessing.
- We drop the ID column, as it does not contain any signal.
- The data contains 1 duplicate row when ignoring the target. We drop this artifact.
- We log scale the target.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="loss",
    problem_type="regression",
    objective_metric_name="mae",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(dataset_mold.path / "train.csv")
print("Loaded data shape:", df.shape)

cat_features = [c for c in list(df) if c.startswith("cat")]
df[cat_features] = df[cat_features].astype("category")
df = df.drop(columns=["id"])

# Drop duplicates w/o target column
df = df.drop_duplicates(subset=[c for c in df.columns if c != task_mold.target_column_name])

# log scale the target
df[task_mold.target_column_name] = np.log(df[task_mold.target_column_name])

df = df.reset_index(drop=True)

Loaded data shape: (188318, 132)


## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 188,317
Columns: 131

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,cat1,cat2,cat3,cat4,cat5,cat6,cat7,cat8,cat9,cat10,cat11,cat12,cat13,cat14,cat15,cat16,cat17,cat18,cat19,cat20,cat21,cat22,cat23,cat24,cat25,cat26,cat27,cat28,cat29,cat30,cat31,cat32,cat33,cat34,cat35,cat36,cat37,cat38,cat39,cat40,cat41,cat42,cat43,cat44,cat45,cat46,cat47,cat48,cat49,cat50,cat51,cat52,cat53,cat54,cat55,cat56,cat57,cat58,cat59,cat60,cat61,cat62,cat63,cat64,cat65,cat66,cat67,cat68,cat69,cat70,cat71,cat72,cat73,cat74,cat75,cat76,cat77,cat78,cat79,cat80,cat81,cat82,cat83,cat84,cat85,cat86,cat87,cat88,cat89,cat90,cat91,cat92,cat93,cat94,cat95,cat96,cat97,cat98,cat99,cat100,cat101,cat102,cat103,cat104,cat105,cat106,cat107,cat108,cat109,cat110,cat111,cat112,cat113,cat114,cat115,cat116,cont1,cont2,cont3,cont4,cont5,cont6,cont7,cont8,cont9,cont10,cont11,cont12,cont13,cont14,loss
0,A,B,A,B,A,A,A,A,B,A,B,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,D,B,B,D,D,B,D,C,B,D,B,A,A,A,A,A,D,B,C,E,A,C,T,B,G,A,A,I,E,G,J,G,BU,BC,C,AS,S,A,O,LB,0.726300,0.245921,0.187583,0.789639,0.310061,0.718367,0.335060,0.30260,0.67135,0.83510,0.569745,0.594646,0.822493,0.714843,7.702186
1,A,B,A,A,A,A,A,A,B,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,D,B,B,D,D,A,B,C,B,D,B,A,A,A,A,A,D,D,C,E,E,D,T,L,F,A,A,E,E,I,K,K,BI,CQ,A,AV,BM,A,O,DP,0.330514,0.737068,0.592681,0.614134,0.885834,0.438917,0.436585,0.60087,0.35127,0.43919,0.338312,0.366307,0.611431,0.304496,7.157424
2,A,B,A,A,B,A,A,A,B,B,B,B,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,D,B,B,B,D,B,D,C,B,B,B,A,A,A,A,A,D,D,C,E,E,A,D,L,O,A,B,E,F,H,F,A,AB,DK,A,C,AF,A,I,GK,0.261841,0.358319,0.484196,0.236924,0.397069,0.289648,0.315545,0.27320,0.26076,0.32446,0.381398,0.373424,0.195709,0.774425,8.008063
3,B,B,A,B,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,D,B,B,D,D,D,B,C,B,D,B,A,A,A,A,A,D,D,C,E,E,D,T,I,D,A,A,E,E,I,K,K,BI,CS,C,N,AE,A,O,DJ,0.321594,0.555782,0.527991,0.373816,0.422268,0.440945,0.391128,0.31796,0.32128,0.44467,0.327915,0.321570,0.605077,0.602642,6.845720
4,A,B,A,B,A,A,A,A,B,B,A,B,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,A,B,A,A,A,A,D,B,D,B,D,B,B,C,B,B,C,A,A,A,B,H,D,B,D,E,E,A,P,F,J,A,A,D,E,K,G,B,H,C,C,Y,BM,A,K,CK,0.273204,0.159990,0.527991,0.473202,0.704268,0.178193,0.247408,0.24564,0.22089,0.21230,0.204687,0.202213,0.246011,0.432606,7.924380


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,cat1,category,0,0.0,2,"A, B"
1,cat2,category,0,0.0,2,"A, B"
2,cat3,category,0,0.0,2,"A, B"
3,cat4,category,0,0.0,2,"A, B"
4,cat5,category,0,0.0,2,"A, B"
5,cat6,category,0,0.0,2,"A, B"
6,cat7,category,0,0.0,2,"A, B"
7,cat8,category,0,0.0,2,"A, B"
8,cat9,category,0,0.0,2,"A, B"
9,cat10,category,0,0.0,2,"A, B"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
cont1,188317.0,0.493861,0.187641,0.000016,0.984975
cont2,188317.0,0.507189,0.207202,0.001149,0.862654
cont3,188317.0,0.498920,0.202104,0.002634,0.944251
cont4,188317.0,0.491810,0.211291,0.176921,0.954297
cont5,188317.0,0.487428,0.209027,0.281143,0.983674
cont6,188317.0,0.490945,0.205273,0.012683,0.997162
cont7,188317.0,0.484971,0.178451,0.069503,1.000000
cont8,188317.0,0.486437,0.199371,0.236880,0.980200
cont9,188317.0,0.485507,0.181661,0.000080,0.995400
cont10,188317.0,0.498066,0.185877,0.000000,0.994980


In [7]:
# Categorical Feature Statistics
cat_stats

value   count    pct
column rank                     
cat1   1        A  141549  75.17
       2        B   46768  24.83
cat10  1        A  160212  85.08
       2        B   28105  14.92
cat100 1        F   42970  22.82
       2        I   39932  21.20
       3        L   19961  10.60
       4        K   13817   7.34
       5        G   12935   6.87
cat101 1        A  106720  56.67
       2        D   17171   9.12
       3        C   16971   9.01
       4        G   10944   5.81
       5        F   10139   5.38
cat102 1        A  177273  94.14
       2        B    5155   2.74
       3        C    4929   2.62
       4        E     482   0.26
       5        D     449   0.24
cat103 1        A  123736  65.71
       2        B   33342  17.71
       3        C   16508   8.77
       4        D    7806   4.15
       5        E    4473   2.38
cat104 1        E   42925  22.79
       2        G   40660  21.59
       3        D   27611  14.66
       4        F   19227  10.21
       5        H   17187   9.13
cat105 1        E   76492  40.62
       2        F   62892  33.40
       3        G   20613  10.95
       4        D   12172   6.46
       5        H   11258   5.98
cat106 1        G   47164  25.05
       2        H   37713  20.03
       3        F   36143  19.19
       4        I   21433  11.38
       5        J   18281   9.71
cat107 1        F   47310  25.12
       2        G   28560  15.17
       3        H   23461  12.46
       4        J   22405  11.90
       5        K   20236  10.75
cat108 1        B   65511  34.79
       2        K   42435  22.53
       3        G   21421  11.37
       4        D   19160  10.17
       5        F   10242   5.44
cat109 1       BI  152918  81.20
       2       AB   21932  11.65
       3       BU    3142   1.67
       4        K    2999   1.59
       5        G    1353   0.72
cat11  1        A  168185  89.31
       2        B   20132  10.69
cat110 1       CL   25305  13.44
       2       EG   24654  13.09
       3       CS   24592  13.06
       4       EB   21395  11.36
       5       CO   17495   9.29
cat111 1        A  128394  68.18
       2        C   32401  17.21
       3        E   14682   7.80
       4        G    7039   3.74
       5        I    3578   1.90
cat112 1        E   25148  13.35
       2       AH   18638   9.90
       3       AS   17669   9.38
       4        J   16222   8.61
       5       AF    9368   4.97
cat113 1       BM   26191  13.91
       2       AE   22030  11.70
       3        L   13058   6.93
       4       AX   12661   6.72
       5        Y   11374   6.04
cat114 1        A  131693  69.93
       2        C   16793   8.92
       3        E   16474   8.75
       4        J    8199   4.35
       5        F    7905   4.20
cat115 1        K   43866  23.29
       2        O   26813  14.24
       3        J   23894  12.69
       4        N   22438  11.92
       5        P   21538  11.44
cat116 1       HK   21061  11.18
       2       DJ   20244  10.75
       3       CK   10162   5.40
       4       DP    9202   4.89
       5       GS    8736   4.64
cat12  1        A  159824  84.87
       2        B   28493  15.13
cat13  1        A  168850  89.66
       2        B   19467  10.34
cat14  1        A  186040  98.79
       2        B    2277   1.21
cat15  1        A  188283  99.98
       2        B      34   0.02
cat16  1        A  181842  96.56
       2        B    6475   3.44
cat17  1        A  187008  99.30
       2        B    1309   0.70
cat18  1        A  187330  99.48
       2        B     987   0.52
cat19  1        A  186509  99.04
       2        B    1808   0.96
cat2   1        A  106720  56.67
       2        B   81597  43.33
cat20  1        A  188113  99.89
       2        B     204   0.11
cat21  1        A  187904  99.78
       2        B     413   0.22
cat22  1        A  188274  99.98
       2        B      43   0.02
cat23  1        A  157444  83.61
       2        B   30873  16.39
cat24  1        A  181976  96.63
       2        B    6341   3.37
cat25  1        A  

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,0.093,-0.312,0.659,0.009,log1p,1156787.0,184996813.6,exponential


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=3, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
019c0a71-9029-727e-a7d9-a4c48238c737
a0ec8284df3fbc11d51def0c245d4d6c501f647a7708984fe7fbba51488a6efe
